In [1]:
# =====================================================
# NDVI STANDARDIZATION (NDVI-A, Z-score)
# Climatological, month-wise, pixel-based
# =====================================================

!pip install rasterio

import os
import numpy as np
import rasterio
from glob import glob
from collections import defaultdict

In [2]:
# =====================================================
# 1. CONFIGURATION
# =====================================================
input_folder = "/content/drive/MyDrive/SmartSDG-Tunisia/Brut_Data/NDVI"
output_folder = "/content/drive/MyDrive/NDVI-A"
os.makedirs(output_folder, exist_ok=True)

MIN_STD = 0.02      # Minimum allowed std (stability)
MIN_MEAN = 0.1      # Vegetation mask threshold
CLIP_Z = 3.0        # Z-score clipping (optional but recommended)

# =====================================================
# 2. GROUP RASTERS BY MONTH
# Expected filename: NDVI_Tunisia_YYYY_MM.tif
# =====================================================
rasters_by_month = defaultdict(list)

for f in sorted(glob(os.path.join(input_folder, "*.tif"))):
    fname = os.path.basename(f).replace(".tif", "")
    parts = fname.split("_")
    try:
        year = parts[-2]
        month = int(parts[-1])
        rasters_by_month[month].append((year, f))
    except:
        print("❌ Ignored:", fname)

# =====================================================
# 3. PROCESS MONTH BY MONTH
# =====================================================
for month in sorted(rasters_by_month.keys()):
    print(f"\n📆 Processing month: {month:02d}")

    files = sorted(rasters_by_month[month], key=lambda x: x[0])

    stack = []
    meta = None

    # -------------------------------------------------
    # Load NDVI stack
    # -------------------------------------------------
    for year, fpath in files:
        with rasterio.open(fpath) as src:
            img = src.read(1).astype(np.float32)
            img[img == src.nodata] = np.nan
            stack.append(img)

            if meta is None:
                meta = src.meta.copy()

    stack = np.stack(stack, axis=0)  # (years, rows, cols)

    # -------------------------------------------------
    # 4. CLIMATOLOGY (ALL YEARS, SAME MONTH)
    # -------------------------------------------------
    mean = np.nanmean(stack, axis=0)
    std  = np.nanstd(stack, axis=0)

    # Stability and vegetation masks
    std[std < MIN_STD] = np.nan
    mean[mean < MIN_MEAN] = np.nan

    # -------------------------------------------------
    # 5. COMPUTE NDVI-A FOR EACH YEAR
    # -------------------------------------------------
    for i, (year, _) in enumerate(files):
        z = (stack[i] - mean) / std

        # Optional clipping
        z = np.clip(z, -CLIP_Z, CLIP_Z)

        meta.update(
            dtype=rasterio.float32,
            count=1,
            nodata=np.nan,
            compress="lzw"
        )

        out_name = f"NDVI_A_{year}_{month:02d}.tif"
        out_path = os.path.join(output_folder, out_name)

        with rasterio.open(out_path, "w", **meta) as dst:
            dst.write(z.astype(np.float32), 1)

        print(f"  ✅ Saved: {out_name}")

print("\n🎉 NDVI-A computation completed successfully.")


📆 Processing month: 01


/tmp/ipython-input-365244062.py:56: RuntimeWarning: Mean of empty slice
  mean = np.nanmean(stack, axis=0)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_nanfunctions_impl.py:2035: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


  ✅ Saved: NDVI_A_2001_01.tif
  ✅ Saved: NDVI_A_2002_01.tif
  ✅ Saved: NDVI_A_2003_01.tif
  ✅ Saved: NDVI_A_2004_01.tif
  ✅ Saved: NDVI_A_2005_01.tif
  ✅ Saved: NDVI_A_2006_01.tif
  ✅ Saved: NDVI_A_2007_01.tif
  ✅ Saved: NDVI_A_2008_01.tif
  ✅ Saved: NDVI_A_2009_01.tif
  ✅ Saved: NDVI_A_2010_01.tif
  ✅ Saved: NDVI_A_2011_01.tif
  ✅ Saved: NDVI_A_2012_01.tif
  ✅ Saved: NDVI_A_2013_01.tif
  ✅ Saved: NDVI_A_2014_01.tif
  ✅ Saved: NDVI_A_2015_01.tif
  ✅ Saved: NDVI_A_2016_01.tif
  ✅ Saved: NDVI_A_2017_01.tif
  ✅ Saved: NDVI_A_2018_01.tif
  ✅ Saved: NDVI_A_2019_01.tif
  ✅ Saved: NDVI_A_2020_01.tif
  ✅ Saved: NDVI_A_2021_01.tif
  ✅ Saved: NDVI_A_2022_01.tif
  ✅ Saved: NDVI_A_2023_01.tif
  ✅ Saved: NDVI_A_2024_01.tif
  ✅ Saved: NDVI_A_2025_01.tif

📆 Processing month: 02
  ✅ Saved: NDVI_A_2000_02.tif
  ✅ Saved: NDVI_A_2001_02.tif
  ✅ Saved: NDVI_A_2002_02.tif
  ✅ Saved: NDVI_A_2003_02.tif
  ✅ Saved: NDVI_A_2004_02.tif
  ✅ Saved: NDVI_A_2005_02.tif
  ✅ Saved: NDVI_A_2006_02.tif
  ✅ Saved: NDVI_

In [3]:
import os
import numpy as np
import rasterio
from glob import glob
from collections import defaultdict

# =====================================================
# CONFIGURATION
# =====================================================
ndvia_folder = "/content/drive/MyDrive/NDVI-A"

# =====================================================
# 1. GROUP NDVI-A RASTERS BY MONTH
# Expected: NDVI_A_YYYY_MM.tif
# =====================================================
rasters_by_month = defaultdict(list)

for f in sorted(glob(os.path.join(ndvia_folder, "*.tif"))):
    fname = os.path.basename(f).replace(".tif", "")
    parts = fname.split("_")
    try:
        year = parts[-2]
        month = int(parts[-1])
        rasters_by_month[month].append((year, f))
    except:
        print("❌ Ignored:", fname)

# =====================================================
# 2. STATISTICS PER MONTH
# =====================================================
print("\n📊 NDVI-A STATISTICS (MONTHLY CLIMATOLOGY CHECK)\n")

for month in sorted(rasters_by_month.keys()):
    values = []

    for year, fpath in rasters_by_month[month]:
        with rasterio.open(fpath) as src:
            arr = src.read(1).astype(float)
            arr[arr == src.nodata] = np.nan
            values.append(arr.flatten())

    values = np.concatenate(values)

    valid = values[~np.isnan(values)]

    if len(valid) == 0:
        print(f"Month {month:02d}: ❌ No valid pixels")
        continue

    print(f"📆 Month {month:02d}")
    print(f"  Min   : {np.nanmin(valid):6.2f}")
    print(f"  Max   : {np.nanmax(valid):6.2f}")
    print(f"  Mean  : {np.nanmean(valid):6.2f}")
    print(f"  Std   : {np.nanstd(valid):6.2f}")
    print(f"  |Z|>3 : {100 * np.sum(np.abs(valid) > 3) / len(valid):5.2f} %")
    print(f"  NaN   : {100 * np.sum(np.isnan(values)) / len(values):5.2f} %\n")

print("✅ NDVI-A verification completed.")


📊 NDVI-A STATISTICS (MONTHLY CLIMATOLOGY CHECK)

📆 Month 01
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   0.99
  |Z|>3 :  0.00 %
  NaN   : 74.04 %

📆 Month 02
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   0.98
  |Z|>3 :  0.00 %
  NaN   : 72.40 %

📆 Month 03
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   0.99
  |Z|>3 :  0.00 %
  NaN   : 72.56 %

📆 Month 04
  Min   :  -3.00
  Max   :   3.00
  Mean  :   0.00
  Std   :   1.00
  |Z|>3 :  0.00 %
  NaN   : 74.78 %

📆 Month 05
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   1.00
  |Z|>3 :  0.00 %
  NaN   : 79.56 %

📆 Month 06
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   0.99
  |Z|>3 :  0.00 %
  NaN   : 84.90 %

📆 Month 07
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   0.99
  |Z|>3 :  0.00 %
  NaN   : 90.78 %

📆 Month 08
  Min   :  -3.00
  Max   :   3.00
  Mean  :  -0.00
  Std   :   0.99
  |Z|>3 :  0.00 %
  NaN   : 92.63 %

📆 Month 09
  Min   :  

In [6]:


import os
import numpy as np
import rasterio
import pandas as pd

 # Dossiers des variables
folder = {

    "NDVI": "/content/drive/MyDrive/NDVI-A",
  }

results = []

def compute_stats(folder_path):
    values = []

    for file in sorted(os.listdir(folder_path)):
        if file.endswith(".tif"):
            raster_path = os.path.join(folder_path, file)

            with rasterio.open(raster_path) as src:
                data = src.read(1).astype("float32")

                # Supprimer NoData
                if src.nodata is not None:
                    data[data == src.nodata] = np.nan

                values.append(data.flatten())

    values = np.concatenate(values)
    values = values[~np.isnan(values)]

    return {
        "Min": np.min(values),
        "Max": np.max(values),
        "Mean": np.mean(values),
        "Std": np.std(values)
    }

# 🔹 Calcul des stats
for var, path in folder.items():
    print(f"📊 Calcul des statistiques pour {var} ...")
    stats = compute_stats(path)
    stats["Variable"] = var
    results.append(stats)

# 🔹 Résultats sous forme de tableau
df_stats = pd.DataFrame(results)
df_stats = df_stats[["Variable", "Min", "Max", "Mean", "Std"]]

print("\n✅ Statistiques descriptives globales :")
print(df_stats)

📊 Calcul des statistiques pour NDVI ...

✅ Statistiques descriptives globales :
  Variable  Min  Max      Mean       Std
0     NDVI -3.0  3.0 -0.002217  0.989818
